In [1]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
subscription_id = "9bf05e2c-7ab6-4705-b7af-45b13a47bfe2"
rg_name = "data-scientist-cert-rg"
workspace_name = "data-scientist-cert"

ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=subscription_id,
    resource_group_name=rg_name,
    workspace_name=workspace_name,
)

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


## Create compute

In [2]:
from azure.ai.ml.entities import AmlCompute

cluster = AmlCompute(
    name="my-cluster",
    type="amlcompute",
    size="STANDARD_D2S_V3",
    min_instances=0,
    max_instances=4,
    idle_time_before_scale_down=120
)

ml_client.begin_create_or_update(cluster).result()

AmlCompute({'type': 'amlcompute', 'created_on': None, 'provisioning_state': 'Succeeded', 'provisioning_errors': None, 'name': 'my-cluster', 'description': None, 'tags': None, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/9bf05e2c-7ab6-4705-b7af-45b13a47bfe2/resourceGroups/data-scientist-cert-rg/providers/Microsoft.MachineLearningServices/workspaces/data-scientist-cert/computes/my-cluster', 'Resource__source_path': '', 'base_path': '/Users/gade/Knowit/DP100/notebooks', 'creation_context': None, 'serialize': <msrest.serialization.Serializer object at 0x16b15d790>, 'resource_id': None, 'location': 'swedencentral', 'size': 'STANDARD_D2S_v3', 'min_instances': 0, 'max_instances': 4, 'idle_time_before_scale_down': 120.0, 'identity': None, 'ssh_public_access_enabled': True, 'ssh_settings': None, 'network_settings': <azure.ai.ml.entities._compute.compute.NetworkSettings object at 0x16afa3f50>, 'tier': 'dedicated', 'enable_node_public_ip': True, 'subnet': None})

# Data Assets

## Download Kaggle data

In [ ]:
import kagglehub
import pandas as pd
from pathlib import Path

def download_data(dataset, path, output):
    dataset_dir = kagglehub.dataset_download(dataset)
    csv_path = f"{dataset_dir}/{path}"

    df = pd.read_csv(csv_path)

    out_dir = Path(output)
    df.to_csv(out_dir, index=False)

kaggle_dataset = "pavansubhasht/ibm-hr-analytics-attrition-dataset"
kaggle_path = "WA_Fn-UseC_-HR-Employee-Attrition.csv"
output_data = "../data/churn-raw.csv"
download_data(dataset=kaggle_dataset, path=kaggle_path, output=output_data)

## URI_FILE

In [9]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

file_asset = Data(
    name="ibm-churn-file",
    path="../data/churn-raw.csv",
    type=AssetTypes.URI_FILE,
    version="1"
)

ml_client.data.create_or_update(file_asset)

Data({'path': 'azureml://subscriptions/9bf05e2c-7ab6-4705-b7af-45b13a47bfe2/resourcegroups/data-scientist-cert-rg/workspaces/data-scientist-cert/datastores/workspaceblobstore/paths/LocalUpload/46f1db532fae41c9b65686f0eb03f6ffc42439cdfbc70a80ebd84c3f90ceae6c/churn-raw.csv', 'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': None, 'type': 'uri_file', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'ibm-churn-file', 'description': None, 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/9bf05e2c-7ab6-4705-b7af-45b13a47bfe2/resourceGroups/data-scientist-cert-rg/providers/Microsoft.MachineLearningServices/workspaces/data-scientist-cert/data/ibm-churn-file/versions/1', 'Resource__source_path': '', 'base_path': '/Users/gade/Knowit/DP100/notebooks', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x301596ea0>, 'serialize': <msrest.serialization.Serializer object at 0x301a3da6

## URI_FOLDER

In [10]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

folder_asset = Data(
    name="ibm-churn-folder",
    path="../data/",
    type=AssetTypes.URI_FOLDER,
    version="1"
)

ml_client.data.create_or_update(folder_asset)

Data({'path': 'azureml://subscriptions/9bf05e2c-7ab6-4705-b7af-45b13a47bfe2/resourcegroups/data-scientist-cert-rg/workspaces/data-scientist-cert/datastores/workspaceblobstore/paths/LocalUpload/794827eb15064df303d933cfaf7dd11c9beb1730f62f350fc0947548ae3ee0a8/data/', 'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': None, 'type': 'uri_folder', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'ibm-churn-folder', 'description': None, 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/9bf05e2c-7ab6-4705-b7af-45b13a47bfe2/resourceGroups/data-scientist-cert-rg/providers/Microsoft.MachineLearningServices/workspaces/data-scientist-cert/data/ibm-churn-folder/versions/1', 'Resource__source_path': '', 'base_path': '/Users/gade/Knowit/DP100/notebooks', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x30195fd40>, 'serialize': <msrest.serialization.Serializer object at 0x3019ad160>

## Data Assets: MLTable

In [2]:
mltable_yml = """
paths:
  - file: ./churn-raw.csv
transformations:
  - read_delimited:
      delimiter: ","
      header: all_files_same_headers
      encoding: utf8
"""

from pathlib import Path
Path("../data/MLTable").write_text(mltable_yml)

150

In [3]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

mltable_asset = Data(
    name="ibm-churn-mltable",
    path="../data/",
    type=AssetTypes.MLTABLE,
    version="1"
)

ml_client.data.create_or_update(mltable_asset)

HttpResponseError: (UserError) A data version with this name and version already exists. If you are trying to create a new data version, use a different name or version. If you are trying to update an existing data version, the existing asset's data uri cannot be changed. Only tags, description, and isArchived can be updated.
Code: UserError
Message: A data version with this name and version already exists. If you are trying to create a new data version, use a different name or version. If you are trying to update an existing data version, the existing asset's data uri cannot be changed. Only tags, description, and isArchived can be updated.
Additional Information:Type: ComponentName
Info: {
    "value": "managementfrontend"
}Type: Correlation
Info: {
    "value": {
        "operation": "57db41289e0f94e7968d8285e3fcf710",
        "request": "0c800b1eb4b9b95c"
    }
}Type: Environment
Info: {
    "value": "swedencentral"
}Type: Location
Info: {
    "value": "swedencentral"
}Type: Time
I

# Datastores

## List datastores

In [11]:
for ds in ml_client.datastores.list():
    print(ds)

account_name: datascientistc5112122140
container_name: training-data
credentials: {}
description: Blob Storage for training data
endpoint: core.windows.net
id: /subscriptions/9bf05e2c-7ab6-4705-b7af-45b13a47bfe2/resourceGroups/data-scientist-cert-rg/providers/Microsoft.MachineLearningServices/workspaces/data-scientist-cert/datastores/blob_training_data
name: blob_training_data
protocol: https
tags: {}
type: azure_blob

account_name: mmstorageswedencentral
container_name: globaldatasets
credentials: {}
endpoint: core.windows.net
id: /subscriptions/9bf05e2c-7ab6-4705-b7af-45b13a47bfe2/resourceGroups/data-scientist-cert-rg/providers/Microsoft.MachineLearningServices/workspaces/data-scientist-cert/datastores/azureml_globaldatasets
name: azureml_globaldatasets
protocol: https
tags: {}
type: azure_blob

account_name: datascientistc5112122140
credentials: {}
endpoint: core.windows.net
file_share_name: azureml-filestore-968e8674-021f-4834-9ad2-cc6c7951a196
id: /subscriptions/9bf05e2c-7ab6-4705

In [6]:
for d in ml_client.data.list():
    print(d.name)
    print(d.type)
    print(d)

diabetes-local
uri_file
creation_context:
  created_at: '2026-01-28T14:14:00.151528+00:00'
  created_by: Morten Gade
  created_by_type: User
  last_modified_at: '2026-01-28T14:14:00.288043+00:00'
latest_version: '1'
name: diabetes-local
properties: {}
tags: {}
type: uri_file

diabetes-datastore-path
uri_folder
creation_context:
  created_at: '2026-01-28T14:17:35.019975+00:00'
  created_by: Morten Gade
  created_by_type: User
  last_modified_at: '2026-01-28T14:17:35.157044+00:00'
latest_version: '1'
name: diabetes-datastore-path
properties: {}
tags: {}
type: uri_folder

diabetes-table
mltable
creation_context:
  created_at: '2026-01-28T14:20:21.817465+00:00'
  created_by: Morten Gade
  created_by_type: User
  last_modified_at: '2026-01-28T14:20:21.960544+00:00'
latest_version: '1'
name: diabetes-table
properties: {}
tags: {}
type: mltable

employee-attrition-csv
uri_file
creation_context:
  created_at: '2026-01-29T12:41:16.164171+00:00'
  created_by: Morten Gade
  created_by_type: User
